In [1]:
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, make_scorer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

In [2]:
# 7 = 5 PCs but with baseline
# 9 = 5 PCs but with 4 clusters
# else = num PC
numPC = 9
d5_coords = np.load(f"/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/DJM_binary_classification/cluster_data/matched_data_{numPC}pc/d5_truncated_coordinates.npy")
d1_coords = np.load(f"/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/DJM_binary_classification/cluster_data/matched_data_{numPC}pc/d1_truncated_coordinates.npy")
cluster_labels = np.load(f"/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/DJM_binary_classification/cluster_data/matched_data_{numPC}pc/cluster_labels.npy")

X_train, X_test, y_train, y_test = train_test_split(
    d5_coords,
    cluster_labels,
    train_size=0.80,
    test_size=0.20,
    random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [3]:
# Define the parameter grid
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'kernel': ['rbf', 'linear', 'poly'],
    'gamma': ['scale', 'auto', 0.1, 1],
    'class_weight': [None, 'balanced'],
    'degree': [2, 3, 4, 5],  # for poly kernel
    'coef0': [0, 0.1, 0.5, 1],  # for poly kernel
}

# Create an SVM classifier
svm = SVC(random_state=42)

# Define a custom scorer that uses F1 score
f1_scorer = make_scorer(f1_score, average='weighted')

# Set up GridSearchCV with StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
random_search = RandomizedSearchCV(svm, param_grid, n_iter=100, cv=skf, 
                                   scoring=f1_scorer, n_jobs=-1, random_state=42)

# Perform the grid search
random_search.fit(X_train_scaled, y_train)

"""
10 - {'kernel': 'rbf', 'gamma': 'scale', 'degree': 5, 'coef0': 0.1, 'class_weight': None, 'C': 10} - 0.7988251282948253
5  - {'kernel': 'linear', 'class_weight': None, 'C': 10} - 0.8545217397255016
3  - {'kernel': 'linear', 'class_weight': None, 'C': 10} - 0.821943847398393
"""

# Print the best parameters and score
print("Best parameters:", random_search.best_params_)
print("Best cross-validation score:", random_search.best_score_)

Best parameters: {'kernel': 'poly', 'gamma': 'scale', 'degree': 3, 'coef0': 0.5, 'class_weight': 'balanced', 'C': 10}
Best cross-validation score: 0.8913779108097291


In [4]:
# Use the best model to make predictions
# best_model = random_search.best_estimator_
best_model = SVC(C=100, kernel='linear', class_weight=None).fit(X_train_scaled, y_train)    # Based on previous best
d5_pred = best_model.predict(X_test_scaled)

d5_accuracy = accuracy_score(y_test, d5_pred)
d5_f1 = f1_score(y_test, d5_pred, average='weighted')

print("Day 5 Results")
print(classification_report(y_test, d5_pred))
print('Accuracy: ', "%.2f" % (d5_accuracy*100))
print('F1: ', "%.2f" % (d5_f1*100))

Day 5 Results
              precision    recall  f1-score   support

           1       0.57      0.80      0.67         5
           2       1.00      0.67      0.80         6
           3       1.00      0.57      0.73         7
           4       0.81      0.91      0.86        23

    accuracy                           0.80        41
   macro avg       0.84      0.74      0.76        41
weighted avg       0.84      0.80      0.80        41

Accuracy:  80.49
F1:  80.34


In [5]:
d1_coords_scaled = scaler.transform(d1_coords)
d1_pred = best_model.predict(d1_coords_scaled)

d1_accuracy = accuracy_score(cluster_labels, d1_pred)
d1_f1 = f1_score(cluster_labels, d1_pred, average='weighted')

print("Day 1 Results")
print(classification_report(cluster_labels, d1_pred))
print('Accuracy: ', "%.2f" % (d1_accuracy*100))
print('F1: ', "%.2f" % (d1_f1*100))

Day 1 Results
              precision    recall  f1-score   support

           1       0.03      0.04      0.04        26
           2       0.14      0.14      0.14        28
           3       0.11      0.06      0.08        34
           4       0.58      0.63      0.61       117

    accuracy                           0.40       205
   macro avg       0.22      0.22      0.21       205
weighted avg       0.37      0.40      0.38       205

Accuracy:  39.51
F1:  38.26


In [6]:
np.random.seed(42)

shuffled_labels = np.random.uniform(1, 5, 205).astype(int)
X_train_shuffled, X_test_shuffled, y_train_shuffled, y_test_shuffled = train_test_split(
    d5_coords,
    shuffled_labels,
    train_size=0.80,
    test_size=0.20,
    random_state=42)

X_train_shuffled_scaled = scaler.fit_transform(X_train_shuffled)
X_test_shuffled_scaled = scaler.transform(X_test_shuffled)

In [7]:
shuffled_model = SVC(C=10, kernel='linear', class_weight=None).fit(X_train_shuffled_scaled, y_train_shuffled)
d5_shuffled_pred = shuffled_model.predict(X_test_shuffled_scaled)

d5_shuffled_accuracy = accuracy_score(y_test_shuffled, d5_shuffled_pred)
d5_shuffled_f1 = f1_score(y_test_shuffled, d5_shuffled_pred, average='weighted')

print("Shuffled Day 5 Results")
print(classification_report(y_test_shuffled, d5_shuffled_pred))
print('Accuracy: ', "%.2f" % (d5_shuffled_accuracy*100))
print('F1: ', "%.2f" % (d5_shuffled_f1*100))


Shuffled Day 5 Results
              precision    recall  f1-score   support

           1       0.29      0.83      0.43        12
           2       0.00      0.00      0.00        10
           3       0.00      0.00      0.00        10
           4       0.00      0.00      0.00         9

    accuracy                           0.24        41
   macro avg       0.07      0.21      0.11        41
weighted avg       0.08      0.24      0.12        41

Accuracy:  24.39
F1:  12.45


/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beh

In [8]:
d1_coords_scaled = scaler.transform(d1_coords)
d1_pred = shuffled_model.predict(d1_coords_scaled)

d1_accuracy = accuracy_score(cluster_labels, d1_pred)
d1_f1 = f1_score(cluster_labels, d1_pred, average='weighted')

print("Shuffled Day 1 Results")
print(classification_report(cluster_labels, d1_pred))
print('Accuracy: ', "%.2f" % (d1_accuracy*100))
print('F1: ', "%.2f" % (d1_f1*100))

Shuffled Day 1 Results
              precision    recall  f1-score   support

           1       0.10      0.54      0.17        26
           2       0.00      0.00      0.00        28
           3       0.00      0.00      0.00        34
           4       0.59      0.23      0.33       117

    accuracy                           0.20       205
   macro avg       0.17      0.19      0.12       205
weighted avg       0.35      0.20      0.21       205

Accuracy:  20.00
F1:  21.02


/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beh

In [9]:
np.random.seed(42)

shuffled_labels = np.random.permutation(cluster_labels)
X_train_shuffled, X_test_shuffled, y_train_shuffled, y_test_shuffled = train_test_split(
    d5_coords,
    shuffled_labels,
    train_size=0.80,
    test_size=0.20,
    random_state=42)

X_train_shuffled_scaled = scaler.fit_transform(X_train_shuffled)
X_test_shuffled_scaled = scaler.transform(X_test_shuffled)

In [10]:
shuffled_model = SVC(C=10, kernel='linear', class_weight=None).fit(X_train_shuffled_scaled, y_train_shuffled)
d5_shuffled_pred = shuffled_model.predict(X_test_shuffled_scaled)

d5_shuffled_accuracy = accuracy_score(y_test_shuffled, d5_shuffled_pred)
d5_shuffled_f1 = f1_score(y_test_shuffled, d5_shuffled_pred, average='weighted')

print("Uniform Labeling Day 5 Results")
print(classification_report(y_test_shuffled, d5_shuffled_pred))
print('Accuracy: ', "%.2f" % (d5_shuffled_accuracy*100))
print('F1: ', "%.2f" % (d5_shuffled_f1*100))

Uniform Labeling Day 5 Results
              precision    recall  f1-score   support

           1       0.00      0.00      0.00         3
           2       0.00      0.00      0.00         6
           3       0.00      0.00      0.00         8
           4       0.59      1.00      0.74        24

    accuracy                           0.59        41
   macro avg       0.15      0.25      0.18        41
weighted avg       0.34      0.59      0.43        41

Accuracy:  58.54
F1:  43.23


/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beh

In [11]:
d1_coords_scaled = scaler.transform(d1_coords)
d1_pred = shuffled_model.predict(d1_coords_scaled)

d1_accuracy = accuracy_score(cluster_labels, d1_pred)
d1_f1 = f1_score(cluster_labels, d1_pred, average='weighted')

print("Uniform Labeling Day 1 Results")
print(classification_report(cluster_labels, d1_pred))
print('Accuracy: ', "%.2f" % (d1_accuracy*100))
print('F1: ', "%.2f" % (d1_f1*100))

Uniform Labeling Day 1 Results
              precision    recall  f1-score   support

           1       0.00      0.00      0.00        26
           2       0.00      0.00      0.00        28
           3       0.00      0.00      0.00        34
           4       0.57      1.00      0.73       117

    accuracy                           0.57       205
   macro avg       0.14      0.25      0.18       205
weighted avg       0.33      0.57      0.41       205

Accuracy:  57.07
F1:  41.48


/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/hades/Desktop/Bruchas Lab/Encoder_Decoder/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beh